# SupplyMind AI — Logistic Regression

Model selection is performed on validation data only.

In [1]:
# -------------------
# Imports
# -------------------

from pathlib import Path

from supplymind.features.predictions.domain.constants import (
    CATEGORICAL_FEATURES,
    NUMERICAL_FEATURES,
)
from supplymind.features.predictions.ml.artifacts import save_model_artifact
# -------------------
# Reload evaluation code
# -------------------

import importlib

import supplymind.features.predictions.ml.evaluation as evaluation

evaluation = importlib.reload(evaluation)

positive_class_probability = evaluation.positive_class_probability
choose_threshold = evaluation.choose_threshold
evaluate_probabilities = evaluation.evaluate_probabilities
BinaryMetrics = evaluation.BinaryMetrics

print("Evaluation module:", evaluation.__file__)
print("BinaryMetrics fields:", BinaryMetrics.__annotations__)

from supplymind.features.predictions.ml.preprocessing import build_preprocessor
from supplymind.features.predictions.ml.reporting import (
    save_evaluation_plots,
    save_feature_importance,
    save_json,
)
from supplymind.features.predictions.ml.training import (
    build_logistic_regression,
    fit_pipeline,
)
from supplymind.features.predictions.ml.workflow import (
    load_clean_syndelay,
    prepare_model_data,
)

Evaluation module: /Users/karima/IronHack/Ironhack-challenges/supplymind-ai/src/supplymind/features/predictions/ml/evaluation.py
BinaryMetrics fields: {'accuracy': 'float', 'precision': 'float', 'recall': 'float', 'f1': 'float', 'roc_auc': 'float', 'average_precision': 'float', 'balanced_accuracy': 'float', 'specificity': 'float', 'true_negative': 'int', 'false_positive': 'int', 'false_negative': 'int', 'true_positive': 'int', 'false_positive_rate': 'float', 'false_negative_rate': 'float', 'threshold': 'float'}


In [2]:
# -------------------
# Project configuration
# -------------------

from pathlib import Path

DATASET_PATH = Path("../data/raw/syndelay/syndelay_v1.csv")
REPORT_ROOT = Path("../reports")
MODEL_ROOT = Path("../models")

assert DATASET_PATH.exists(), (
    f"Dataset not found at {DATASET_PATH}. "
    "Place syndelay_v1.csv under data/raw/syndelay/."
)

In [3]:
# -------------------
# Prepare identical model data
# -------------------

df = load_clean_syndelay(DATASET_PATH)
data = prepare_model_data(df)

In [4]:
# -------------------
# Build preprocessing
# -------------------

preprocessor = build_preprocessor(
    NUMERICAL_FEATURES,
    CATEGORICAL_FEATURES,
    scale_numerical=True,
)

In [5]:
# -------------------
# Train model
# -------------------

estimator = build_logistic_regression()
model = fit_pipeline(
    preprocessor,
    estimator,
    data.X_train,
    data.y_train,
)

In [6]:
# -------------------
# Validation probabilities
# -------------------

validation_probability = positive_class_probability(
    model,
    data.X_validation,
)

threshold, threshold_search = choose_threshold(
    data.y_validation,
    validation_probability,
)

print("Selected threshold:", threshold)
threshold_search.sort_values(
    [
        "balanced_accuracy",
        "f1",
        "recall",
    ],
    ascending=[
        False,
        False,
        False,
    ],
).head(10)

Selected threshold: 0.34000000000000014


,accuracy,precision,recall,f1,roc_auc,average_precision,balanced_accuracy,specificity,true_negative,false_positive,false_negative,true_positive,false_positive_rate,false_negative_rate,threshold
28,0.691721,0.879569,0.539816,0.669030,0.740043,0.834546,0.719457,0.899097,8866,995,6195,7267,0.100903,0.460184,0.48
29,0.691549,0.880248,0.538924,0.668540,0.740043,0.834546,0.719417,0.899909,8874,987,6207,7255,0.100091,0.461076,0.49
30,0.691335,0.880544,0.538256,0.668111,0.740043,0.834546,0.719285,0.900314,8878,983,6216,7246,0.099686,0.461744,0.50
31,0.691163,0.880764,0.537736,0.667774,0.740043,0.834546,0.719177,0.900619,8881,980,6223,7239,0.099381,0.462264,0.51
42,0.690863,0.881406,0.536622,0.667098,0.740043,0.834546,0.719026,0.901430,8889,972,6238,7224,0.098570,0.463378,0.62
40,0.690863,0.881313,0.536696,0.667128,0.740043,0.834546,0.719012,0.901328,8888,973,6237,7225,0.098672,0.463304,0.60
38,0.690863,0.881220,0.536770,0.667159,0.740043,0.834546,0.718999,0.901227,8887,974,6236,7226,0.098773,0.463230,0.58
32,0.690949,0.880692,0.537364,0.667466,0.740043,0.834546,0.718992,0.900619,8881,980,6228,7234,0.099381,0.462636,0.52
41,0.690820,0.881298,0.536622,0.667067,0.740043,0.834546,0.718975,0.901328,8888,973,6238,7224,0.098672,0.463378,0.61
43,0.690777,0.881470,0.536399,0.666944,0.740043,0.834546,0.718965,0.901531,8890,971,6241,7221,0.098469,0.463601,0.63


In [7]:
# -------------------
# Validation metrics
# -------------------

metrics = evaluate_probabilities(
    data.y_validation,
    validation_probability,
    threshold=threshold,
)

metrics.to_dict()

{'accuracy': 0.646357672683617,
 'precision': 0.6739855846235985,
 'recall': 0.7501857079185856,
 'f1': 0.710047106798847,
 'roc_auc': 0.740042948190666,
 'average_precision': 0.8345463668153176,
 'balanced_accuracy': 0.6273999222079492,
 'specificity': 0.5046141364973127,
 'true_negative': 4976,
 'false_positive': 4885,
 'false_negative': 3363,
 'true_positive': 10099,
 'false_positive_rate': 0.4953858635026874,
 'false_negative_rate': 0.24981429208141434,
 'threshold': 0.34000000000000014}

In [8]:
# -------------------
# Save candidate reports
# -------------------

MODEL_NAME = "logistic_regression"
REPORT_DIR = REPORT_ROOT / "models" / MODEL_NAME

save_json(
    metrics.to_dict(),
    REPORT_DIR / "validation_metrics.json",
)
threshold_search.to_csv(
    REPORT_DIR / "threshold_search.csv",
    index=False,
)
save_evaluation_plots(
    data.y_validation,
    validation_probability,
    threshold,
    REPORT_DIR,
    "validation",
)
save_feature_importance(
    model,
    REPORT_DIR / "feature_importance",
)

save_model_artifact(
    model,
    {
        "model_name": MODEL_NAME,
        "model_version": "0.1.0-candidate",
        "threshold": threshold,
        "validation_metrics": metrics.to_dict(),
    },
    MODEL_ROOT / "candidates" / MODEL_NAME,
)